# 01 — Data Understanding & EDA

In [ ]:
import pandas as pd 
import numpy as np
import sqlite3
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
db_path = r"../data/inventory.db"
conn = sqlite3.connect(db_path)
print("Database connected successfully!")

tables = pd.read_sql_query("select name from sqlite_master where type = 'table'", conn)

In [ ]:
tables

In [ ]:
for table in tables['name']:
    print('Table name', table)
    df = pd.read_sql_query(f"select * from {table} limit 5", conn)
    display(df)

In [ ]:
vendor_df = pd.read_sql_query("select * from vendor_invoice", conn)
print(vendor_df.shape)
vendor_df.head()

In [ ]:
vendor_df.info()

In [ ]:
#  Missing-value count and percentage for all vendor_df columns
missing_count = vendor_df.isnull().sum()
missing_pct   = vendor_df.isnull().mean() * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %':     missing_pct.round(2)
})
print('vendor_df shape:', vendor_df.shape)
print()
print(missing_summary.to_string())

In [ ]:
vendor_df[['Quantity', 'Freight', 'Dollars']].corr()

### Key Insights: Relationship Between Quantity, Dollars, and Freight
* **Extremely Strong Positive Correlation:** The heatmap reveals near-perfect linear correlations among all three variables (all correlation coefficients are between $0.95$ and $0.99$).
* **Primary Cost Drivers:** As order **Quantity** and total **Dollars** increase, **Freight** costs scale up almost proportionately. 
* **Model Implication:** Because Dollars and Quantity share such a high multicollinearity with Freight, they serve as powerful, reliable predictors for estimating freight costs in downstream regression models.

In [ ]:
# Relationship between Quantity, Dollars and Freight
# Correlation heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(vendor_df[['Quantity', 'Dollars', 'Freight']].corr(), annot=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:
plt.scatter(vendor_df['Quantity'],vendor_df['Freight'], color = '#f57a55')
plt.scatter(vendor_df['Dollars'],vendor_df['Freight'], color = '#7f1e5a')
plt.legend(['Quantity','Dollars'])
plt.ylabel('Freight Cost')
plt.show()

### Key Takeaways from Scatter Plot Analysis
* **Direct Scaling with Freight:** Both **Quantity** and total **Dollars** show a strong, consistent upward linear relationship with **Freight Cost**.
* **Visual Confirmation of Cost Drivers:** Heavier orders and higher invoice totals require greater shipping overhead, confirming them as reliable predictors for estimating freight expenses.

In [ ]:
vendor_df['freight_per_unit'] = np.where(
    vendor_df['Quantity'] > 0,
    vendor_df['Freight'] / vendor_df['Quantity'],
    np.nan
)

In [ ]:
low_quantity = vendor_df['Quantity'].quantile(0.25)
high_quantity = vendor_df['Quantity'].quantile(0.75)

print("Low Quantity:", low_quantity)
print("High Quantity:", high_quantity)

In [ ]:
cost_lq = vendor_df.loc[vendor_df['Quantity']<low_quantity,'freight_per_unit'].mean()
print("Average Freight cost when quantity is low:",cost_lq)

In [ ]:
cost_hq = vendor_df.loc[vendor_df['Quantity']>high_quantity,'freight_per_unit'].mean()
print("Average Freight cost when quantity is high:",cost_hq)

In [ ]:
#  Distribution histograms for Dollars, Freight and Quantity
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, colour in zip(axes,
                            ['Dollars', 'Freight', 'Quantity'],
                            ['#4c72b0', '#dd8452', '#55a868']):
    ax.hist(vendor_df[col].dropna(), bins=50, color=colour, edgecolor='white', linewidth=0.5)
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    skew = vendor_df[col].skew()
    ax.annotate(f'Skewness: {skew:.2f}', xy=(0.97, 0.95),
                xycoords='axes fraction', ha='right', va='top',
                fontsize=9, color='black',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

plt.suptitle('Histograms: Dollars, Freight and Quantity', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### What This Chart Tells Us
* **Clustered Data:** The tall spikes on the left show that most of our invoice orders have smaller amounts, freight costs, and quantities.
* **High Right Skewness:** The skewness values (around 4.46 to 4.66) prove that the data is heavily stretched to the right by a small number of very large, high-value bulk orders.

In [ ]:
#  Boxplots for Dollars, Freight and Quantity (outlier visualisation)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, colour in zip(axes,
                            ['Dollars', 'Freight', 'Quantity'],
                            ['#4c72b0', '#dd8452', '#55a868']):
    ax.boxplot(vendor_df[col].dropna(),
               patch_artist=True,
               boxprops=dict(facecolor=colour, alpha=0.6),
               medianprops=dict(color='black', linewidth=2),
               flierprops=dict(marker='o', markersize=2, alpha=0.3))
    ax.set_title(f'{col} Boxplot')
    ax.set_ylabel(col)
    ax.set_xticks([])

plt.suptitle('Boxplots: Dollars, Freight and Quantity (whiskers = 1.5 x IQR)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Handling Outliers
* Decision: Retain all extreme values as-is.

* Justification: In invoice and supply chain datasets, high-value amounts, heavy freight, and large quantities represent legitimate business operations (e.g., bulk corporate orders or seasonal shipments) rather than data entry errors.


In [ ]:
# Monthly EDA using InvoiceDate

eda_df = vendor_df.copy()
eda_df['InvoiceDate'] = pd.to_datetime(eda_df['InvoiceDate'], errors='coerce')
eda_df['YearMonth'] = eda_df['InvoiceDate'].dt.to_period('M')

monthly_counts  = eda_df.groupby('YearMonth').size().rename('Invoice Count')
monthly_freight = eda_df.groupby('YearMonth')['Freight'].mean().rename('Avg Freight ($)')

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Monthly invoice counts
axes[0].bar(monthly_counts.index.astype(str), monthly_counts.values,
            color='#4c72b0', edgecolor='white', linewidth=0.5)
axes[0].set_ylabel('Invoice Count')
axes[0].set_title('Monthly Invoice Counts')
axes[0].tick_params(axis='x', rotation=45)

# Monthly average freight
axes[1].bar(monthly_freight.index.astype(str), monthly_freight.values,
            color='#dd8452', edgecolor='white', linewidth=0.5)
axes[1].set_ylabel('Average Freight ($)')
axes[1].set_title('Monthly Average Freight Cost')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Print monthly summary table
monthly_summary = pd.DataFrame({'Invoice Count': monthly_counts, 'Avg Freight ($)': monthly_freight.round(2)})
print(monthly_summary.to_string())

### Key Takeaways from Monthly EDA
* **Consistent Transaction Volumes:** Monthly invoice counts remain relatively stable throughout 2024, generally averaging between 400 and 500 transactions per month.
* **Dataset Time Boundary:** January 2025 shows a sharp decline (140 invoices), indicating that the dataset's time coverage ends early in that month.
* **Freight Cost Trends:** Average monthly freight costs fluctuate between roughly  210 dollors and 380 dollors with noticeable peaks occurring during the mid and late summer/winter months.

In [ ]:
X = vendor_df[["Dollars"]]
y = vendor_df["Freight"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Convert InvoiceDate to datetime objects
vendor_df["InvoiceDate"] = pd.to_datetime(vendor_df["InvoiceDate"])


cutoff = pd.Timestamp("2024-02-01")

train = vendor_df["InvoiceDate"] < cutoff
test = vendor_df["InvoiceDate"] >= cutoff

X_train = X.loc[train]
X_test = X.loc[test]
y_train = y.loc[train]
y_test = y.loc[test]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

if X_train.empty or X_test.empty:
    raise ValueError("Training or testing data is empty. Check the cutoff date.")